In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn
import streamlit as st

print("Welcome to SkinMatch!")

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Streamlit:", st.__version__)

Welcome to SkinMatch!
Python: 3.11.16
NumPy: 2.4.6
Pandas: 3.0.6
Scikit-learn: 1.9.1
Streamlit: 1.64.0


# SkinMatch: Ingredient-Based Skincare Analysis and Skin Type Recommendation System

## 1. Project Introduction

SkinMatch is a Data Science project that investigates how skincare
ingredients and product characteristics relate to consumer recommendations
across different skin types.

The project uses Sephora product information and consumer reviews to
analyze skincare formulations, identify recommendation patterns, and
develop an ingredient-based product recommendation system.

The final objective is to create an interactive application that allows
users to search for a skincare product and explore its ingredients,
consumer recommendation statistics, and similar products.

The application will compare results across four skin types:

- Dry
- Oily
- Normal
- Combination

Users will not need to provide their own skin type.

### Research Question

Can machine learning predict whether a skincare product will be
recommended by consumers with a specific skin type based on its
ingredient profile?

### Project Objectives

1. Clean and integrate the Sephora product and review datasets.
2. Explore consumer recommendation patterns across skin types.
3. Extract and engineer features from skincare ingredient lists.
4. Develop and evaluate machine learning classification models.
5. Compare observed recommendation rates for each product and skin type.
6. Develop an ingredient-based product similarity system.
7. Build an interactive SkinMatch application using Streamlit.

### Dataset Source

Sephora Products and Skincare Reviews

Source: Kaggle — nadyinky/sephora-products-and-skincare-reviews

### Tools and Technologies

Python, pandas, NumPy, scikit-learn, Matplotlib, Seaborn,
Jupyter Notebook, VS Code, and Streamlit.

In [2]:
# CELL 1 — Dataset File Verification

from pathlib import Path
import pandas as pd

# Find the project directory regardless of whether
# VS Code starts the notebook in the root or notebooks folder.

current_directory = Path.cwd()

if (current_directory / "data" / "raw").exists():
    project_path = current_directory

elif (current_directory.parent / "data" / "raw").exists():
    project_path = current_directory.parent

else:
    raise FileNotFoundError(
        "Could not locate the SkinMatch project directory."
    )

raw_data_path = project_path / "data" / "raw"

csv_files = sorted(raw_data_path.glob("*.csv"))

print("=" * 60)
print("SKINMATCH — DATASET VERIFICATION")
print("=" * 60)

print("\nProject directory:")
print(project_path)

print("\nRaw data directory:")
print(raw_data_path)

print("\nCSV files found:", len(csv_files))

for file in csv_files:

    size_mb = file.stat().st_size / (1024 ** 2)

    print(f"{file.name} — {size_mb:.2f} MB")

if not csv_files:
    print("\nNo CSV files were found. Check the data/raw folder.")

SKINMATCH — DATASET VERIFICATION

Project directory:
/Users/flortrejorodriguez/Skinmatch final

Raw data directory:
/Users/flortrejorodriguez/Skinmatch final/data/raw

CSV files found: 6
product_info.csv — 7.55 MB
reviews_0-250.csv — 269.18 MB
reviews_1250-end.csv — 23.07 MB
reviews_250-500.csv — 95.65 MB
reviews_500-750.csv — 53.70 MB
reviews_750-1250.csv — 55.29 MB


In [3]:
# CELL 2 — Initial Dataset Inspection

dataset_overview = []

for file in csv_files:

    sample = pd.read_csv(
        file,
        nrows=3
    )

    dataset_overview.append({
        "File": file.name,
        "Number of Columns": len(sample.columns),
        "Column Names": ", ".join(sample.columns)
    })

overview_df = pd.DataFrame(dataset_overview)

display(overview_df)

,File,Number of Columns,Column Names
0,product_info.csv,27,"product_id, product_name, brand_id, brand_name..."
1,reviews_0-250.csv,19,"Unnamed: 0, author_id, rating, is_recommended,..."
2,reviews_1250-end.csv,19,"Unnamed: 0, author_id, rating, is_recommended,..."
3,reviews_250-500.csv,19,"Unnamed: 0, author_id, rating, is_recommended,..."
4,reviews_500-750.csv,19,"Unnamed: 0, author_id, rating, is_recommended,..."
5,reviews_750-1250.csv,19,"Unnamed: 0, author_id, rating, is_recommended,..."


## 2. Data Cleaning and Preparation

The original Sephora dataset contains product information and consumer reviews
distributed across multiple CSV files.

This stage prepares a skincare-specific dataset by:

- Selecting skincare products with ingredient information.
- Combining the original review files.
- Retaining reviews with valid skin types and recommendation labels.
- Creating product-level consumer recommendation statistics.

The resulting datasets will support machine learning, ingredient analysis,
and the SkinMatch application.

In [4]:
# CELL 3 — Prepare SkinMatch Dataset

import pandas as pd

# --------------------------------------------------
# 1. LOAD PRODUCT INFORMATION
# --------------------------------------------------

products = pd.read_csv(
    raw_data_path / "product_info.csv",
    low_memory=False
)

print("=" * 60)
print("SKINMATCH — DATA PREPARATION")
print("=" * 60)

print("\nOriginal products:", len(products))

print("\nProduct categories:")
print(products["primary_category"].value_counts())


# --------------------------------------------------
# 2. SELECT SKINCARE PRODUCTS
# --------------------------------------------------

products["product_id"] = (
    products["product_id"].astype("string").str.strip()
)

skincare_products = products[
    products["primary_category"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("skincare")
    .fillna(False)
].copy()

if skincare_products.empty:
    raise ValueError(
        "No skincare products found. Check primary_category."
    )

print("\nSkincare products:", len(skincare_products))


# --------------------------------------------------
# 3. KEEP PRODUCTS WITH INGREDIENT INFORMATION
# --------------------------------------------------

valid_ingredients = (
    skincare_products["ingredients"]
    .astype("string")
    .str.strip()
    .notna()
    &
    skincare_products["ingredients"]
    .astype("string")
    .str.strip()
    .ne("")
    .fillna(False)
    &
    skincare_products["ingredients"]
    .astype("string")
    .str.strip()
    .ne("[]")
    .fillna(False)
)

skincare_products = skincare_products[
    valid_ingredients
].copy()

skincare_products = skincare_products.drop_duplicates(
    subset="product_id"
)

skincare_products["product_category"] = (
    skincare_products["tertiary_category"]
    .fillna(skincare_products["secondary_category"])
)

product_columns = [
    "product_id",
    "product_name",
    "brand_name",
    "product_category",
    "price_usd",
    "ingredients"
]

skincare_products = skincare_products[
    product_columns
].copy()

print(
    "Skincare products with ingredients:",
    len(skincare_products)
)


# --------------------------------------------------
# 4. LOAD REVIEW FILES
# --------------------------------------------------

review_files = sorted(
    raw_data_path.glob("reviews_*.csv")
)

required_review_columns = [
    "product_id",
    "skin_type",
    "is_recommended",
    "rating"
]

review_parts = []

valid_product_ids = set(
    skincare_products["product_id"]
)

for file in review_files:

    print(f"\nReading: {file.name}")

    for chunk in pd.read_csv(
        file,
        usecols=required_review_columns,
        dtype={"product_id": "string"},
        chunksize=100_000
    ):

        chunk["product_id"] = (
            chunk["product_id"].str.strip()
        )

        chunk = chunk[
            chunk["product_id"].isin(
                valid_product_ids
            )
        ]

        if not chunk.empty:
            review_parts.append(chunk)


if not review_parts:
    raise ValueError(
        "No matching skincare reviews found."
    )

reviews = pd.concat(
    review_parts,
    ignore_index=True
)

del review_parts

print(
    "\nMatching skincare reviews:",
    len(reviews)
)


# --------------------------------------------------
# 5. CLEAN REVIEW DATA
# --------------------------------------------------

reviews["skin_type"] = (
    reviews["skin_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

reviews["is_recommended"] = pd.to_numeric(
    reviews["is_recommended"],
    errors="coerce"
)

reviews["rating"] = pd.to_numeric(
    reviews["rating"],
    errors="coerce"
)

valid_skin_types = [
    "dry",
    "oily",
    "normal",
    "combination"
]

reviews = reviews[
    reviews["skin_type"].isin(valid_skin_types)
    &
    reviews["is_recommended"].isin([0, 1])
].copy()

reviews["is_recommended"] = (
    reviews["is_recommended"].astype(int)
)

print("\nCleaned reviews:", len(reviews))

print("\nReviews by skin type:")
print(reviews["skin_type"].value_counts())


# --------------------------------------------------
# 6. CALCULATE CONSUMER RECOMMENDATION STATISTICS
# --------------------------------------------------

consumer_evidence = (
    reviews
    .groupby(
        ["product_id", "skin_type"]
    )
    .agg(
        review_count=("is_recommended", "size"),
        positive_reviews=("is_recommended", "sum"),
        average_rating=("rating", "mean")
    )
    .reset_index()
)

consumer_evidence["recommendation_percentage"] = (
    consumer_evidence["positive_reviews"]
    / consumer_evidence["review_count"]
    * 100
).round(2)


# --------------------------------------------------
# 7. BUILD APPLICATION DATABASE
# --------------------------------------------------

skinmatch_database = consumer_evidence.merge(
    skincare_products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print(
    "\nProduct-skin type combinations:",
    len(skinmatch_database)
)

print(
    "Products with valid reviews:",
    skinmatch_database["product_id"].nunique()
)


# --------------------------------------------------
# 8. SAVE PROCESSED DATASETS
# --------------------------------------------------

processed_path = project_path / "data" / "processed"

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

skincare_products.to_csv(
    processed_path / "skincare_products.csv",
    index=False
)

reviews.to_csv(
    processed_path / "skincare_reviews.csv",
    index=False
)

skinmatch_database.to_csv(
    processed_path / "skinmatch_database.csv",
    index=False
)

print("\nSUCCESS — PROCESSED FILES SAVED!")

print("1. skincare_products.csv")
print("2. skincare_reviews.csv")
print("3. skinmatch_database.csv")

SKINMATCH — DATA PREPARATION

Original products: 8494

Product categories:
primary_category
Skincare           2420
Makeup             2369
Hair               1464
Fragrance          1432
Bath & Body         405
Mini Size           288
Men                  60
Tools & Brushes      52
Gifts                 4
Name: count, dtype: int64

Skincare products: 2420
Skincare products with ingredients: 2286

Reading: reviews_0-250.csv

Reading: reviews_1250-end.csv

Reading: reviews_250-500.csv

Reading: reviews_500-750.csv

Reading: reviews_750-1250.csv

Matching skincare reviews: 1072386

Cleaned reviews: 893263

Reviews by skin type:
skin_type
combination    493728
dry            171386
normal         120188
oily           107961
Name: count, dtype: int64[pyarrow]

Product-skin type combinations: 8388
Products with valid reviews: 2223

SUCCESS — PROCESSED FILES SAVED!
1. skincare_products.csv
2. skincare_reviews.csv
3. skinmatch_database.csv


## 3. Ingredient Feature Engineering

Ingredient information is essential to the SkinMatch recommendation system.

The original ingredient lists are stored as text. They must be converted
into structured lists before they can be used in machine learning or
ingredient similarity calculations.

The preprocessing procedure extracts ingredient names, standardizes
their formatting, and preserves chemical names containing commas.

The resulting ingredient features will help analyze relationships
between product formulations and consumer recommendations.

In [5]:
# CELL 4 — Ingredient Extraction and Feature Engineering

import ast
import re
import numpy as np

# --------------------------------------------------
# 1. INGREDIENT PARSER
# --------------------------------------------------

def parse_ingredients(value):

    if pd.isna(value):
        return []

    text = str(value).strip()

    # Convert Python-style ingredient lists into actual lists
    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, list):
            items = parsed
        else:
            items = [text]

    except (ValueError, SyntaxError):
        items = [text]

    ingredients = []

    for item in items:

        if not isinstance(item, str):
            continue

        # Split ingredient lists without breaking chemical
        # names such as 1,2-Hexanediol.

        parts = re.split(
            r",\s+(?=[A-Za-z(])",
            item
        )

        for ingredient in parts:

            ingredient = ingredient.strip()

            ingredient = re.sub(
                r"\s+",
                " ",
                ingredient
            )

            ingredient = ingredient.lower()

            if ingredient:
                ingredients.append(ingredient)

    return list(dict.fromkeys(ingredients))


# --------------------------------------------------
# 2. APPLY INGREDIENT EXTRACTION
# --------------------------------------------------

skincare_products["ingredient_list"] = (
    skincare_products["ingredients"]
    .apply(parse_ingredients)
)

skincare_products["ingredient_count"] = (
    skincare_products["ingredient_list"]
    .apply(len)
)


# --------------------------------------------------
# 3. INGREDIENT ANALYSIS
# --------------------------------------------------

print("=" * 60)
print("SKINMATCH — INGREDIENT FEATURE ENGINEERING")
print("=" * 60)

print(
    "\nProducts processed:",
    len(skincare_products)
)

print(
    "Average ingredients per product:",
    round(skincare_products["ingredient_count"].mean(), 2)
)

print(
    "Median ingredients per product:",
    skincare_products["ingredient_count"].median()
)

print(
    "Products with no extracted ingredients:",
    (skincare_products["ingredient_count"] == 0).sum()
)


# --------------------------------------------------
# 4. DISPLAY EXAMPLE
# --------------------------------------------------

example = skincare_products.iloc[0]

print("\nEXAMPLE PRODUCT:")
print(example["product_name"])

print("\nEXTRACTED INGREDIENTS:")

for ingredient in example["ingredient_list"][:20]:
    print("-", ingredient)


# --------------------------------------------------
# 5. SAVE PROCESSED PRODUCT INFORMATION
# --------------------------------------------------

skincare_products[
    [
        "product_id",
        "product_name",
        "brand_name",
        "product_category",
        "price_usd",
        "ingredients",
        "ingredient_count"
    ]
].to_csv(
    processed_path / "skincare_products.csv",
    index=False
)

print("\nIngredient extraction completed.")

SKINMATCH — INGREDIENT FEATURE ENGINEERING

Products processed: 2286
Average ingredients per product: 36.16
Median ingredients per product: 33.0
Products with no extracted ingredients: 0

EXAMPLE PRODUCT:
GENIUS Sleeping Collagen Moisturizer

EXTRACTED INGREDIENTS:
- collagen (vegan)*
- water (aqua
- eau)
- ethylhexyl palmitate
- oryza sativa (rice) bran extract
- caprylic/capric triglyceride
- glycerin
- cetearyl methicone
- dimethicone
- cetearyl alcohol
- pyrus malus (apple) fruit extract
- chlorella protothecoides oil
- polysorbate 60
- glyceryl glucoside
- polyglyceryl-2 stearate
- parachlorella beijerinckii exopolysaccharides
- collagen amino acids (vegan)*
- ceramide np
- silybum marianum fruit extract
- helianthus annuus (sunflower) extract

Ingredient extraction completed.


In [6]:
# CELL 4 — Corrected Ingredient Extraction

import ast
import re
import json
import pandas as pd


# --------------------------------------------------
# 1. SPLIT INGREDIENTS SAFELY
# --------------------------------------------------

def split_ingredient_text(text):

    ingredients = []
    current = []
    parentheses_depth = 0

    for i, char in enumerate(text):

        if char == "(":
            parentheses_depth += 1

        elif char == ")":
            parentheses_depth = max(0, parentheses_depth - 1)

        if char == "," and parentheses_depth == 0:

            previous = text[i - 1] if i > 0 else ""
            remaining = text[i + 1:].lstrip()
            following = remaining[0] if remaining else ""

            # Preserve chemical names such as 1,2-Hexanediol
            if previous.isdigit() and following.isdigit():
                current.append(char)
                continue

            ingredient = "".join(current).strip()

            if ingredient:
                ingredients.append(ingredient)

            current = []

        else:
            current.append(char)

    last_ingredient = "".join(current).strip()

    if last_ingredient:
        ingredients.append(last_ingredient)

    return ingredients


# --------------------------------------------------
# 2. PARSE AND NORMALIZE INGREDIENTS
# --------------------------------------------------

def parse_ingredients(value):

    if pd.isna(value):
        return []

    text = str(value).strip()

    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, list):
            items = parsed
        else:
            items = [text]

    except (ValueError, SyntaxError):
        items = [text]

    ingredients = []

    for item in items:

        if not isinstance(item, str):
            continue

        parts = split_ingredient_text(item)

        for ingredient in parts:

            ingredient = re.sub(
                r"\s+",
                " ",
                ingredient.strip().lower()
            )

            # Normalize common water names
            if ingredient in [
                "water",
                "water (aqua)",
                "water (aqua, eau)",
                "water/aqua/eau",
                "aqua",
                "aqua (water)"
            ]:
                ingredient = "water"

            if ingredient:
                ingredients.append(ingredient)

    # Remove duplicates while preserving order
    return list(dict.fromkeys(ingredients))


# --------------------------------------------------
# 3. APPLY CORRECTED PARSER
# --------------------------------------------------

skincare_products["ingredient_list"] = (
    skincare_products["ingredients"]
    .apply(parse_ingredients)
)

skincare_products["ingredient_count"] = (
    skincare_products["ingredient_list"]
    .apply(len)
)


# --------------------------------------------------
# 4. VERIFY INGREDIENT EXTRACTION
# --------------------------------------------------

print("=" * 60)
print("SKINMATCH — CORRECTED INGREDIENT EXTRACTION")
print("=" * 60)

print("\nProducts processed:", len(skincare_products))

print(
    "Average ingredients per product:",
    round(skincare_products["ingredient_count"].mean(), 2)
)

print(
    "Median ingredients per product:",
    skincare_products["ingredient_count"].median()
)

print(
    "Products with no extracted ingredients:",
    (skincare_products["ingredient_count"] == 0).sum()
)

print("\nEXAMPLE PRODUCT:")

example = skincare_products.iloc[0]

print(example["product_name"])

print("\nFIRST 15 INGREDIENTS:")

for ingredient in example["ingredient_list"][:15]:
    print("-", ingredient)


# --------------------------------------------------
# 5. SAVE CORRECTED INGREDIENT DATA
# --------------------------------------------------

skincare_products["ingredient_list_json"] = (
    skincare_products["ingredient_list"]
    .apply(json.dumps)
)

skincare_products[
    [
        "product_id",
        "product_name",
        "brand_name",
        "product_category",
        "price_usd",
        "ingredients",
        "ingredient_list_json",
        "ingredient_count"
    ]
].to_csv(
    processed_path / "skincare_products.csv",
    index=False
)

print("\nSUCCESS — CORRECTED INGREDIENT DATA SAVED!")

SKINMATCH — CORRECTED INGREDIENT EXTRACTION

Products processed: 2286
Average ingredients per product: 36.84
Median ingredients per product: 34.0
Products with no extracted ingredients: 0

EXAMPLE PRODUCT:
GENIUS Sleeping Collagen Moisturizer

FIRST 15 INGREDIENTS:
- collagen (vegan)*
- water
- ethylhexyl palmitate
- oryza sativa (rice) bran extract
- caprylic/capric triglyceride
- glycerin
- cetearyl methicone
- dimethicone
- cetearyl alcohol
- pyrus malus (apple) fruit extract
- chlorella protothecoides oil
- polysorbate 60
- glyceryl glucoside
- polyglyceryl-2 stearate
- parachlorella beijerinckii exopolysaccharides

SUCCESS — CORRECTED INGREDIENT DATA SAVED!


## 4. Machine Learning Dataset Preparation

The supervised learning task is to predict whether a consumer recommends
a skincare product based on skin type, product category, and ingredients.

Reviews are grouped by product and skin type to reduce memory usage.
The number of positive and negative recommendations is retained so the
model can be trained using sample weights.

Products are divided into training and test sets using a group-based
split. This prevents reviews of the same product from appearing in both
sets.

The final test set will be evaluated once, after the modeling approach
has been fixed.

In [9]:
# CELL 5 — Prepare Machine Learning Data

import numpy as np
import pandas as pd

from collections import Counter
from scipy import sparse

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer


# --------------------------------------------------
# 1. GROUP REVIEWS BY PRODUCT AND SKIN TYPE
# --------------------------------------------------

ml_data = (
    reviews
    .groupby(["product_id", "skin_type"])
    .agg(
        review_count=("is_recommended", "size"),
        positive_reviews=("is_recommended", "sum")
    )
    .reset_index()
)

ml_data["negative_reviews"] = (
    ml_data["review_count"]
    - ml_data["positive_reviews"]
)


# --------------------------------------------------
# 2. ADD PRODUCT INFORMATION
# --------------------------------------------------

ml_data = ml_data.merge(
    skincare_products[
        [
            "product_id",
            "product_category",
            "ingredient_list"
        ]
    ],
    on="product_id",
    how="inner",
    validate="many_to_one"
)

ml_data["product_category"] = (
    ml_data["product_category"]
    .fillna("Unknown")
    .astype(str)
)


# --------------------------------------------------
# 3. SPLIT DATA BY PRODUCT
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        ml_data,
        groups=ml_data["product_id"]
    )
)

train_data = ml_data.iloc[train_idx].copy()

test_data = ml_data.iloc[test_idx].copy()


# --------------------------------------------------
# 4. VERIFY NO PRODUCT OVERLAP
# --------------------------------------------------

train_products = set(train_data["product_id"])

test_products = set(test_data["product_id"])

product_overlap = train_products.intersection(
    test_products
)

assert len(product_overlap) == 0, (
    "Data leakage: products overlap!"
)


# --------------------------------------------------
# 5. SELECT INGREDIENT VOCABULARY
#    USING TRAINING PRODUCTS ONLY
# --------------------------------------------------

training_product_ingredients = (
    train_data
    .drop_duplicates("product_id")["ingredient_list"]
)

ingredient_frequency = Counter()

for ingredient_list in training_product_ingredients:

    ingredient_frequency.update(
        set(ingredient_list)
    )

# Keep frequently observed ingredients.
# Maximum of 500 ingredient features.

ingredient_vocabulary = [
    ingredient
    for ingredient, count
    in ingredient_frequency.most_common()
    if count >= 10
][:500]

ingredient_vocabulary = sorted(
    ingredient_vocabulary
)

ingredient_vocabulary_set = set(
    ingredient_vocabulary
)


# --------------------------------------------------
# 6. ENCODE INGREDIENTS
# --------------------------------------------------

ingredient_encoder = MultiLabelBinarizer(
    classes=ingredient_vocabulary,
    sparse_output=True
)

# Initialize the encoder using the vocabulary
# selected from training products only.

ingredient_encoder.fit(
    [ingredient_vocabulary]
)

def encode_ingredients(data):

    filtered_lists = [
        [
            ingredient
            for ingredient in ingredient_list
            if ingredient in ingredient_vocabulary_set
        ]
        for ingredient_list in data["ingredient_list"]
    ]

    return ingredient_encoder.transform(
        filtered_lists
    )


X_train_ingredients = encode_ingredients(
    train_data
)

X_test_ingredients = encode_ingredients(
    test_data
)


# --------------------------------------------------
# 7. ENCODE CATEGORICAL FEATURES
# --------------------------------------------------

categorical_columns = [
    "skin_type",
    "product_category"
]

category_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_categories = category_encoder.fit_transform(
    train_data[categorical_columns]
)

X_test_categories = category_encoder.transform(
    test_data[categorical_columns]
)


# --------------------------------------------------
# 8. COMBINE FEATURES
# --------------------------------------------------

X_train = sparse.hstack(
    [
        X_train_categories,
        X_train_ingredients
    ],
    format="csr"
)

X_test = sparse.hstack(
    [
        X_test_categories,
        X_test_ingredients
    ],
    format="csr"
)


# --------------------------------------------------
# 9. DISPLAY SUMMARY
# --------------------------------------------------

print("=" * 60)
print("SKINMATCH — MACHINE LEARNING DATASET")
print("=" * 60)

print(
    "\nTraining products:",
    len(train_products)
)

print(
    "Test products:",
    len(test_products)
)

print(
    "\nTraining reviews:",
    train_data["review_count"].sum()
)

print(
    "Test reviews:",
    test_data["review_count"].sum()
)

print(
    "\nIngredient features:",
    len(ingredient_vocabulary)
)

print(
    "Total model features:",
    X_train.shape[1]
)

print(
    "\nTraining matrix:",
    X_train.shape
)

print(
    "Test matrix:",
    X_test.shape
)

print(
    "\nOverlapping products:",
    len(product_overlap)
)

print("\nSUCCESS — ML DATASET READY!")

SKINMATCH — MACHINE LEARNING DATASET

Training products: 1778
Test products: 445

Training reviews: 690993
Test reviews: 202270

Ingredient features: 500
Total model features: 538

Training matrix: (6691, 538)
Test matrix: (1697, 538)

Overlapping products: 0

SUCCESS — ML DATASET READY!


## 5. Logistic Regression

Logistic Regression is used to investigate whether skin type,
product category, and ingredient information can predict consumer
recommendations.

Because the dataset contains multiple reviews for the same product
and skin type, positive and negative review counts are used as
sample weights.

The model is evaluated on products that were excluded from training.

A baseline based on the training recommendation rate is included
to determine whether the model provides predictive value beyond
the overall proportion of positive recommendations.

In [10]:
# CELL 6 — Train and Evaluate Logistic Regression

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix
)

import joblib


# --------------------------------------------------
# 1. PREPARE WEIGHTED TRAINING DATA
# --------------------------------------------------

positive_mask = (
    train_data["positive_reviews"].to_numpy() > 0
)

negative_mask = (
    train_data["negative_reviews"].to_numpy() > 0
)

X_train_weighted = sparse.vstack(
    [
        X_train[positive_mask],
        X_train[negative_mask]
    ],
    format="csr"
)

y_train_weighted = np.concatenate(
    [
        np.ones(positive_mask.sum(), dtype=int),
        np.zeros(negative_mask.sum(), dtype=int)
    ]
)

training_weights = np.concatenate(
    [
        train_data.loc[
            positive_mask,
            "positive_reviews"
        ].to_numpy(),

        train_data.loc[
            negative_mask,
            "negative_reviews"
        ].to_numpy()
    ]
)

# Normalize weights while preserving their relative values.

training_weights = (
    training_weights
    * len(training_weights)
    / training_weights.sum()
)


# --------------------------------------------------
# 2. TRAIN LOGISTIC REGRESSION
# --------------------------------------------------

logistic_model = LogisticRegression(
    C=0.01,
    solver="liblinear",
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_weighted,
    y_train_weighted,
    sample_weight=training_weights
)


# --------------------------------------------------
# 3. PREPARE TEST LABELS AND WEIGHTS
# --------------------------------------------------

test_positive_mask = (
    test_data["positive_reviews"].to_numpy() > 0
)

test_negative_mask = (
    test_data["negative_reviews"].to_numpy() > 0
)

y_test = np.concatenate(
    [
        np.ones(test_positive_mask.sum(), dtype=int),
        np.zeros(test_negative_mask.sum(), dtype=int)
    ]
)

test_weights = np.concatenate(
    [
        test_data.loc[
            test_positive_mask,
            "positive_reviews"
        ].to_numpy(),

        test_data.loc[
            test_negative_mask,
            "negative_reviews"
        ].to_numpy()
    ]
)


# --------------------------------------------------
# 4. GENERATE MODEL PREDICTIONS
# --------------------------------------------------

test_probabilities = (
    logistic_model.predict_proba(X_test)[:, 1]
)

model_probabilities = np.concatenate(
    [
        test_probabilities[test_positive_mask],
        test_probabilities[test_negative_mask]
    ]
)


# --------------------------------------------------
# 5. BASELINE
# --------------------------------------------------

training_recommendation_rate = (
    train_data["positive_reviews"].sum()
    /
    train_data["review_count"].sum()
)

baseline_probabilities = np.full(
    len(y_test),
    training_recommendation_rate
)


# --------------------------------------------------
# 6. EVALUATION FUNCTION
# --------------------------------------------------

def evaluate_model(
    model_name,
    probabilities
):

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    return {
        "Model": model_name,

        "Accuracy": accuracy_score(
            y_test,
            predictions,
            sample_weight=test_weights
        ),

        "Balanced Accuracy": balanced_accuracy_score(
            y_test,
            predictions,
            sample_weight=test_weights
        ),

        "Precision": precision_score(
            y_test,
            predictions,
            sample_weight=test_weights,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test,
            predictions,
            sample_weight=test_weights,
            zero_division=0
        ),

        "F1": f1_score(
            y_test,
            predictions,
            sample_weight=test_weights,
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_test,
            probabilities,
            sample_weight=test_weights
        ),

        "Average Precision": average_precision_score(
            y_test,
            probabilities,
            sample_weight=test_weights
        ),

        "Brier Score": brier_score_loss(
            y_test,
            probabilities,
            sample_weight=test_weights
        )
    }


# --------------------------------------------------
# 7. COMPARE RESULTS
# --------------------------------------------------

results = pd.DataFrame(
    [
        evaluate_model(
            "Baseline",
            baseline_probabilities
        ),

        evaluate_model(
            "Logistic Regression",
            model_probabilities
        )
    ]
)

print("=" * 60)
print("SKINMATCH — FINAL MODEL EVALUATION")
print("=" * 60)

display(
    results.round(4)
)


# --------------------------------------------------
# 8. CONFUSION MATRIX
# --------------------------------------------------

model_predictions = (
    model_probabilities >= 0.5
).astype(int)

cm = confusion_matrix(
    y_test,
    model_predictions,
    sample_weight=test_weights
)

print("\nCONFUSION MATRIX:")

print(cm)


# --------------------------------------------------
# 9. SAVE MODEL
# --------------------------------------------------

model_path = project_path / "models"

model_path.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    {
        "model": logistic_model,
        "category_encoder": category_encoder,
        "ingredient_encoder": ingredient_encoder,
        "ingredient_vocabulary": ingredient_vocabulary,
        "categorical_columns": categorical_columns,
        "training_recommendation_rate":
            training_recommendation_rate
    },
    model_path / "skinmatch_logistic_model.joblib"
)

results.to_csv(
    processed_path / "model_results.csv",
    index=False
)

print("\nSUCCESS — MODEL AND RESULTS SAVED!")

SKINMATCH — FINAL MODEL EVALUATION


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,Average Precision,Brier Score
0,Baseline,0.8478,0.5,0.8478,1.0,0.9176,0.5000,0.8478,0.1291
1,Logistic Regression,0.8478,0.5,0.8478,1.0,0.9176,0.5712,0.8783,0.1283



CONFUSION MATRIX:
[[     0  30780]
 [     0 171490]]

SUCCESS — MODEL AND RESULTS SAVED!


### Model Interpretation

Logistic Regression achieved an ROC-AUC of 0.5712 on previously unseen
products, compared with 0.5000 for the baseline.

However, at the default classification threshold of 0.5, the model
predicted every review as a positive recommendation.

The identical accuracy of 84.78% reflects the class imbalance in the
dataset rather than strong classification performance.

These results suggest that the available ingredient, skin type, and
product category features provide limited predictive information about
individual consumer recommendations.

Therefore, SkinMatch will not use the classifier to make definitive
skin suitability claims.

The application will instead combine observed consumer recommendation
statistics with an ingredient-based product similarity system.

## 6. Ingredient-Based Recommendation Engine

The SkinMatch recommendation engine uses ingredient profiles to identify
skincare products with similar formulations.

Each product is represented using its extracted ingredient list.

MultiLabelBinarizer converts the ingredients into numerical features,
and TF-IDF weighting reduces the influence of ingredients shared by
many products.

Cosine Similarity measures similarity between product ingredient vectors.

Recommendations are restricted to products within the same skincare
category to avoid comparing unrelated product types.

Ingredient similarity describes formulation overlap. It does not
establish equivalent concentration, safety, effectiveness, or
dermatological suitability.

In [11]:
# CELL 7 — Ingredient-Based Recommendation Engine

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity


# --------------------------------------------------
# 1. PREPARE PRODUCT CATALOG
# --------------------------------------------------

catalog = (
    skincare_products[
        [
            "product_id",
            "product_name",
            "brand_name",
            "product_category",
            "ingredient_list"
        ]
    ]
    .drop_duplicates("product_id")
    .reset_index(drop=True)
    .copy()
)

catalog["product_category"] = (
    catalog["product_category"]
    .fillna("Uncategorized")
    .astype(str)
)


# --------------------------------------------------
# 2. CONVERT INGREDIENTS INTO NUMERICAL FEATURES
# --------------------------------------------------

similarity_encoder = MultiLabelBinarizer(
    sparse_output=True
)

ingredient_binary_matrix = (
    similarity_encoder.fit_transform(
        catalog["ingredient_list"]
    )
)


# --------------------------------------------------
# 3. APPLY TF-IDF WEIGHTING
# --------------------------------------------------

tfidf_transformer = TfidfTransformer(
    norm="l2"
)

ingredient_tfidf_matrix = (
    tfidf_transformer.fit_transform(
        ingredient_binary_matrix
    )
)


# --------------------------------------------------
# 4. CALCULATE SIMILARITY WITHIN PRODUCT CATEGORIES
# --------------------------------------------------

recommendation_records = []

for category, indices in catalog.groupby(
    "product_category"
).indices.items():

    indices = np.asarray(indices)

    if len(indices) < 2:
        continue

    category_matrix = (
        ingredient_tfidf_matrix[indices]
    )

    similarity_matrix = cosine_similarity(
        category_matrix
    )

    # Prevent a product from recommending itself.

    np.fill_diagonal(
        similarity_matrix,
        -1
    )

    for row_position, product_index in enumerate(indices):

        scores = similarity_matrix[row_position]

        top_positions = (
            np.argsort(scores)[::-1][:5]
        )

        for position in top_positions:

            similarity_score = float(
                scores[position]
            )

            if similarity_score <= 0:
                continue

            similar_product_index = (
                indices[position]
            )

            recommendation_records.append(
                {
                    "product_id":
                        catalog.loc[
                            product_index,
                            "product_id"
                        ],

                    "similar_product_id":
                        catalog.loc[
                            similar_product_index,
                            "product_id"
                        ],

                    "similarity_score":
                        similarity_score,

                    "product_category":
                        category
                }
            )


# --------------------------------------------------
# 5. CREATE RECOMMENDATION DATABASE
# --------------------------------------------------

similar_products = pd.DataFrame(
    recommendation_records,
    columns=[
        "product_id",
        "similar_product_id",
        "similarity_score",
        "product_category"
    ]
)

similar_products = similar_products.sort_values(
    [
        "product_id",
        "similarity_score"
    ],
    ascending=[True, False]
)


# --------------------------------------------------
# 6. SAVE RECOMMENDATIONS
# --------------------------------------------------

similar_products.to_csv(
    processed_path / "ingredient_similarities.csv",
    index=False
)


# --------------------------------------------------
# 7. SAVE SIMILARITY ENGINE
# --------------------------------------------------

joblib.dump(
    {
        "encoder": similarity_encoder,
        "tfidf_transformer": tfidf_transformer,
        "ingredient_matrix": ingredient_tfidf_matrix,
        "product_ids": catalog["product_id"].tolist()
    },
    model_path / "ingredient_similarity.joblib"
)


# --------------------------------------------------
# 8. DISPLAY RESULTS
# --------------------------------------------------

print("=" * 60)
print("SKINMATCH — INGREDIENT SIMILARITY ENGINE")
print("=" * 60)

print(
    "\nProducts analyzed:",
    len(catalog)
)

print(
    "Unique ingredient features:",
    len(similarity_encoder.classes_)
)

print(
    "Ingredient matrix:",
    ingredient_tfidf_matrix.shape
)

print(
    "Similar-product recommendations:",
    len(similar_products)
)

print("\nEXAMPLE RECOMMENDATIONS:")

example_product_id = (
    similar_products.iloc[0]["product_id"]
)

example_product = catalog[
    catalog["product_id"] == example_product_id
].iloc[0]

print(
    "\nOriginal product:",
    example_product["product_name"]
)

example_recommendations = (
    similar_products[
        similar_products["product_id"]
        == example_product_id
    ]
    .merge(
        catalog[
            [
                "product_id",
                "product_name",
                "brand_name"
            ]
        ],
        left_on="similar_product_id",
        right_on="product_id",
        how="left",
        suffixes=("", "_matched")
    )
)

display(
    example_recommendations[
        [
            "product_name",
            "brand_name",
            "similarity_score"
        ]
    ].round(3)
)

print(
    "\nSUCCESS — SIMILARITY ENGINE SAVED!"
)

SKINMATCH — INGREDIENT SIMILARITY ENGINE

Products analyzed: 2286
Unique ingredient features: 8177
Ingredient matrix: (2286, 8177)
Similar-product recommendations: 11083

EXAMPLE RECOMMENDATIONS:

Original product: Renewing Eye Cream


,product_name,brand_name,similarity_score
0,Shaba Complex Firming Eye Serum,Drunk Elephant,0.212
1,Retinol Youth Renewal Eye Masks,Murad,0.191
2,GENIUS Ultimate Anti-Aging Eye Cream,Algenist,0.177
3,Age Reversal Eye Complex,Dermalogica,0.177
4,Retinol Youth Renewal Eye Serum,Murad,0.174



SUCCESS — SIMILARITY ENGINE SAVED!


## 7. Final Results and Discussion

### Project Summary

SkinMatch was developed using the Sephora Products and Skincare Reviews
dataset to investigate relationships between skincare ingredients,
skin types, and consumer recommendations.

The project integrates three components:

1. Supervised Machine Learning research.
2. Consumer recommendation analysis across skin types.
3. An ingredient-based product recommendation engine.

### Data Preparation

The original dataset contained 8,494 products, including 2,420
products classified as skincare.

After filtering products with ingredient information, 2,286 skincare
products were retained for ingredient analysis.

The supervised learning dataset included 893,263 valid skincare
reviews associated with 2,223 products.

The data was divided by product to prevent the same product from
appearing in both training and testing sets.

### Machine Learning Results

Logistic Regression was trained using:

- Skin type
- Product category
- 500 ingredient features

The model was evaluated on 445 previously unseen skincare products.

| Metric | Baseline | Logistic Regression |
|---|---:|---:|
| Accuracy | 0.8478 | 0.8478 |
| Balanced Accuracy | 0.5000 | 0.5000 |
| ROC-AUC | 0.5000 | 0.5712 |
| Average Precision | 0.8478 | 0.8783 |
| Brier Score | 0.1291 | 0.1283 |

Although Logistic Regression achieved a higher ROC-AUC than the
baseline, it predicted all test reviews as positive recommendations
at the default classification threshold.

The results indicate limited predictive performance for unseen
products using the available features.

Consequently, the classifier was not used to generate consumer-facing
skin suitability recommendations.

### Ingredient-Based Recommendation Engine

A content-based recommendation system was developed using TF-IDF
and Cosine Similarity.

Skincare products were represented numerically using their
extracted ingredient profiles.

TF-IDF reduced the relative importance of ingredients shared
by many products, while Cosine Similarity identified products
with similar ingredient representations.

Recommendations were restricted to products within the same
skincare category.

This system was integrated into the SkinMatch application to
help consumers discover products with similar ingredient profiles.

Ingredient similarity does not establish equivalent product
effectiveness, safety, or suitability for a specific skin type.

## 8. Conclusion

The SkinMatch project investigated whether machine learning could
predict consumer skincare recommendations using skin type,
product category, and ingredient information.

The supervised learning experiment demonstrated that these
features provided limited predictive information for previously
unseen skincare products.

This finding influenced the final application design.

Rather than presenting weak predictions as reliable recommendations,
SkinMatch uses observed Sephora consumer feedback to compare
recommendation patterns across four skin types.

The application also incorporates a content-based recommendation
engine using TF-IDF and Cosine Similarity to identify products
with similar ingredient profiles.

The final result is an interactive, installable mobile web application
that integrates Data Science techniques into a practical skincare
product analysis experience.

### Project Limitations

- The dataset contains historical Sephora consumer reviews.
- Skin types are self-reported by reviewers.
- Consumer recommendations do not establish clinical effectiveness.
- Ingredient lists do not provide complete formulation concentrations.
- Ingredient similarity does not guarantee similar product performance.
- The supervised learning model showed limited generalization.
- The dataset does not contain verified UPC/EAN barcode identifiers.
- The application uses a static dataset rather than live Sephora data.

### Future Improvements

1. Integrate verified barcode-to-product identification.
2. Expand the product catalog.
3. Improve ingredient normalization and synonym recognition.
4. Incorporate additional validated ingredient information.
5. Explore more informative product and consumer features.
6. Investigate improved prediction methods using future datasets.

### Final Deliverable

SkinMatch is available as an installable Progressive Web App (PWA).

The application allows consumers to search for skincare products,
compare observed recommendation rates across four skin types,
examine ingredient lists, and discover similar formulations.

The Data Science methodology and machine learning results are
documented separately in this notebook.

## 9. SkinMatch Application

The final SkinMatch application is publicly available through
GitHub Pages.

**Live Application:**

https://flortr3.github.io/skinmatch-app/

**GitHub Repository:**

https://github.com/flortr3/skinmatch-app

The application was developed as a Progressive Web App (PWA)
and can be added to the home screen of a compatible mobile device.

The application interface uses HTML, CSS, and JavaScript,
while the underlying data processing and recommendation engine
were developed using Python and scikit-learn.